# MultimodalFusion

> Notebook generated from [`examples/multimodal/05_multimodal_fusion.py`](../../examples/multimodal/05_multimodal_fusion.py).

| Field | Value |
|------|-------|
| **Dataset** | VQA-style (embedded, 3 scenes) |
| **API key** | ✅ **No API keys required** — uses stubs/mocks. |

**Expected result:** Fusion under concat / moderator / moa per scene.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
MultimodalFusion — Combine observations by modality (concat | moderator | moa)
====================================================================================
Component: SPEC-MM-AGT-005 / prismal.agents.multimodal.multimodal_fusion

Dataset: VQA-style — IMG + AUDIO + TEXT observations over the same scene
  • Inspired by Visual Question Answering 2.0 (Goyal et al. 2017),
    where the correct answer requires reasoning over image + text.
  • Here each scene comes with 3 contributions (vision, audio, text);
    fusion synthesizes a single answer.
  • Why: VQA is the canonical benchmark for multimodal fusion and
    covers the 3 modes of the component.

Component description:
  `MultimodalFusion(strategy, moa, moderator_fn, settings).combine(
    contributions, context=…)` produces a `FusionResult(answer,
    contributions, strategy_used)` using one of three strategies:

    - "concat"     : deterministic, labels each contribution with
                     `[modality · agent_id · conf=…]` and concatenates.
                     Zero LLM calls — ideal for tests / fallback.
    - "moderator"  : a single call to a multimodal LLM (via
                     injectable `moderator_fn`). Synthesizes a single
                     coherent paragraph.
    - "moa"        : delegates to `MixtureOfAgents` — N proposers +
                     aggregator. Higher quality, more cost.

Usage:
    uv run python examples/multimodal/05_multimodal_fusion.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

from dataclasses import dataclass

from prismal.agents.multimodal import (
    Modality,
    ModalContribution,
    MultimodalFusion,
)

## Dataset: 3 VQA-style scenes

In [ ]:
SCENES = [
    {
        "id": "vqa_001",
        "question": "What is the dog doing on the beach?",
        "contributions": [
            ModalContribution(
                modality=Modality.IMAGE,
                content="A golden retriever running across wet sand near the shoreline.",
                agent_id="vision",
                confidence=0.92,
            ),
            ModalContribution(
                modality=Modality.AUDIO,
                content="Sound of waves crashing and faint barking.",
                agent_id="audio",
                confidence=0.78,
            ),
            ModalContribution(
                modality=Modality.TEXT,
                content="The user uploaded this photo with caption 'Max's morning run'.",
                agent_id="text",
                confidence=0.85,
            ),
        ],
    },
    {
        "id": "vqa_002",
        "question": "Why is the crowd cheering?",
        "contributions": [
            ModalContribution(
                modality=Modality.IMAGE,
                content="A soccer player celebrating with arms raised, ball in the net.",
                agent_id="vision",
                confidence=0.88,
            ),
            ModalContribution(
                modality=Modality.AUDIO,
                content="Stadium chants and a sharp referee whistle.",
                agent_id="audio",
                confidence=0.81,
            ),
            ModalContribution(
                modality=Modality.VIDEO,
                content="Player kicks ball past the goalkeeper at second 14.",
                agent_id="video",
                confidence=0.90,
            ),
        ],
    },
    {
        "id": "vqa_003",
        "question": "Is the meeting room available?",
        "contributions": [
            ModalContribution(
                modality=Modality.IMAGE,
                content="Conference room with empty chairs and no people visible.",
                agent_id="vision",
                confidence=0.94,
            ),
            ModalContribution(
                modality=Modality.TEXT,
                content="Calendar API reports no events between 10:00 and 12:00.",
                agent_id="text",
                confidence=0.99,
            ),
        ],
    },
]

## Fake moderator — synthesizes from the prompt without calling an LLM

In [ ]:
async def fake_moderator(prompt: str) -> str:
    """Takes the contributions from the prompt and produces a stable summary."""
    # In production this would be: `llm.ainvoke(messages).content`
    contributions = prompt.split("\n\n")[1]   # block "[modality · agent · …]"
    points = [
        line.split("\n", 1)[1]
        for line in contributions.split("\n\n")
        if "\n" in line and line.startswith("[")
    ]
    return "Synthesised: " + " ".join(points)

## Fake MoA (duck-typed) without LLMs

In [ ]:
# MultimodalFusion._combine_moa only needs `await moa.generate(prompt, state)`
# and `result.final_answer`. We build a lightweight stand-in for the demo
# (the real `MixtureOfAgents` requires model ids configured in the provider
# registry, which isn't applicable without API keys).
@dataclass
class _FakeMoAResult:
    final_answer: str


class FakeMoA:
    """Duck-typed stand-in for MixtureOfAgents for the `moa` fusion."""

    def __init__(self, n_proposers: int = 3) -> None:
        self._n = n_proposers

    async def generate(self, query: str, state) -> _FakeMoAResult:  # noqa: ARG002
        proposals = [f"[Expert {i + 1}] view of: {query[:60]}" for i in range(self._n)]
        synth = (
            f"MoA-synth ({self._n} proposers): integrated answer drawing from "
            + "; ".join(p.split(": ", 1)[1] for p in proposals)
        )
        return _FakeMoAResult(final_answer=synth)


def make_fake_moa() -> FakeMoA:
    """3 synthetic proposers + mock aggregator."""
    return FakeMoA(n_proposers=3)


async def main() -> None:
    print("=" * 70)
    print("MultimodalFusion · 3 strategies over VQA-style scenes")
    print("=" * 70)

    for scene in SCENES:
        print("\n" + "─" * 70)
        print(f"Scene {scene['id']} · {scene['question']}")
        print("─" * 70)

        for strategy in ("concat", "moderator", "moa"):
            kwargs = {}
            if strategy == "moderator":
                kwargs["moderator_fn"] = fake_moderator
            if strategy == "moa":
                kwargs["moa"] = make_fake_moa()
            fusion = MultimodalFusion(strategy=strategy, **kwargs)
            result = await fusion.combine(
                scene["contributions"], context=scene["question"]
            )
            print(f"\n  [{strategy}] · strategy_used={result.strategy_used}")
            for line in result.answer.splitlines()[:6]:
                print(f"    {line}")
            if len(result.answer.splitlines()) > 6:
                print(f"    … ({len(result.answer.splitlines()) - 6} more lines)")

    # Extra demo: error if the strategy is unknown
    print("\n" + "─" * 70)
    print("Validation: an unknown strategy raises ValueError on construction")
    print("─" * 70)
    try:
        MultimodalFusion(strategy="ensemble")  # type: ignore[arg-type]
    except ValueError as exc:
        print(f"  ✓ ValueError: {exc}")

    print("\n" + "=" * 70)
    print("OK — fusion covers concat (no LLM) / moderator (1 LLM) / MoA (N+1)")
    print("=" * 70)


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()